# Lab 5: Model Context Protocol (MCP) with LangChain & OpenRouter

Welcome to **Lab 5** of the Agentic AI series! In this lab, you will explore the **Model Context Protocol (MCP)**—an open standard created by Anthropic that enables AI models and agents to securely connect to external tools, data sources, and servers over standardized communication protocols (such as Streamable HTTP or STDIO).

### Key Concepts Covered:
1. **MCP Server**: Hosting tools over HTTP (`streamable-http`) using `mcp.server.mcpserver.MCPServer`.
2. **MCP Adapter**: Using `langchain.mcp.MCPAdapter` to dynamically discover tools at runtime.
3. **Dynamic Agent Binding**: Binding discovered MCP tools directly into a LangChain agent using `create_agent`.
4. **Agentic Reasoning & Execution**: Allowing the LLM to inspect, decide, and invoke MCP tools on a remote server to answer queries.

## 1. Environment & API Key Configuration

Load your OpenRouter API key and MCP server URL from `.env`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "http://localhost:8000/mcp")
LLM_MODEL = os.getenv("LLM_MODEL", "nvidia/nemotron-3-super-120b-a12b:free")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file. Please check lab5/.env.")

print("✓ OpenRouter API key loaded successfully.")
print(f"✓ MCP Server URL: {MCP_SERVER_URL}")
print(f"✓ Default Model : {LLM_MODEL}")

## 2. Verify MCP Server Availability

> **Note:** Make sure you have started the MCP server in a separate terminal before running this notebook:
> ```bash
> cd lab5
> python server.py
> ```

In [ ]:
import httpx

base_url = MCP_SERVER_URL.split("/mcp")[0]
try:
    resp = httpx.get(base_url, timeout=3.0)
    print(f"✓ MCP Server is active and reachable at {base_url} (HTTP {resp.status_code})")
except Exception as err:
    print(f"✗ Error: Could not reach MCP Server at {base_url}: {err}")
    print("Please start the server with: python server.py")

## 3. Connect to MCP Server & Dynamically Discover Tools

Using `langchain.mcp.MCPAdapter`, the client initiates a session with the MCP server, queries the protocol's tool registry, and converts remote tools into LangChain `BaseTool` objects automatically.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Required for nested asyncio loops in Jupyter

from langchain.mcp import MCPAdapter

async def get_tools():
    async with MCPAdapter(MCP_SERVER_URL) as adapter:
        return await adapter.list_tools()

import asyncio
tools = asyncio.run(get_tools())

print(f"Discovered {len(tools)} tools from '{MCP_SERVER_URL}':\n")
for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool.name}")
    print(f"   Description: {tool.description}")
    print(f"   Arguments  : {tool.args}\n")

## 4. Initialize LLM via OpenRouter

We use `ChatOpenRouter` with our chosen model.

In [ ]:
try:
    from langchain_openrouter import ChatOpenRouter
    model = ChatOpenRouter(
        model=LLM_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0
    )
except ImportError:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(
        model=LLM_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0
    )

print(f"✓ Chat model initialized: {model.model_name}")

## 5. Create LangChain Agent with MCP Tools

We compile an agent using `create_agent(model=model, tools=tools)` from LangChain / LangGraph.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools
)

print("✓ Agent created and bound to MCP tools!")

## 6. Execute Agent Queries Over MCP

Let's test the agent on various student records scenarios.

In [ ]:
async def ask_agent(query: str):
    print(f"User Query: {query}\n" + "-" * 50)
    async with MCPAdapter(MCP_SERVER_URL) as adapter:
        # Re-use dynamic session
        active_tools = await adapter.list_tools()
        active_agent = create_agent(model=model, tools=active_tools)
        response = await active_agent.ainvoke({
            "messages": [{"role": "user", "content": query}]
        })
        answer = response["messages"][-1].content
        print(f"Agent Response:\n{answer}\n" + "=" * 50)
        return answer

In [ ]:
# Query 1: Single subject marks retrieval
await ask_agent("What are Praveen's AI marks?")

In [ ]:
# Query 2: Multiple tool execution (attendance + average)
await ask_agent("What is Ravi's attendance and what is his average marks across all subjects?")

In [ ]:
# Query 3: Multi-student comparative reasoning
await ask_agent("Compare the python marks of Ravi and Praveen, and tell me who scored higher.")

## 7. Summary & Benefits of MCP

### Why Model Context Protocol (MCP)?
1. **Decoupled Architecture**: Tools run as independent microservices. The agent doesn't need local code access to execute them.
2. **Dynamic Discovery**: The agent learns what tools are available and their schemas on-the-fly at runtime.
3. **Security & Governance**: Tool execution happens in the server's controlled environment with boundary checks.
4. **Standardization**: Whether connected to a local server or a remote enterprise system, the interaction adheres to an open, standardized protocol.